# Desmatamento na Amazônia Legal por município, estado e região

Calcula a área desmatada (km² e %) em cada um dos 773 municípios da Amazônia Legal a partir do PRODES/INPE (Amazônia e Cerrado), separando o desmatamento **histórico** (acumulado até 2007) e **recente** (2008–2025, também ano a ano), e agrega por estado e para a região. Exporta planilha, CSV e GeoPackage.

Autor: Murilo Henderson Costa · [github.com/murilohenderson/desmatamento-amazonia-legal](https://github.com/murilohenderson/desmatamento-amazonia-legal)

## 1. Configuração
Ajuste a pasta do Drive e os nomes dos arquivos, se necessário.

In [ ]:
!pip install -q pyogrio
import os, glob, shutil, time, zipfile
import numpy as np, shapely, pyogrio
import geopandas as gpd, pandas as pd
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/'   # pasta onde estão os dados (busca em subpastas)
B = '/content/dados/'
OUT = '/content/resultados/'
os.makedirs(B, exist_ok=True); os.makedirs(OUT, exist_ok=True)

## 2. Cópia dos dados do Drive

In [ ]:
def copiar(nome):
    achados = glob.glob(DRIVE + '**/' + nome, recursive=True)
    if not achados:
        raise FileNotFoundError(f'{nome} não encontrado no Drive')
    origem = achados[0]
    if nome.endswith('.shp'):
        for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
            if os.path.exists(origem[:-4] + ext):
                shutil.copy(origem[:-4] + ext, B)
    elif not os.path.exists(B + nome):
        shutil.copy(origem, B)
    print('Copiado:', origem)
    return B + nome

MUNICIPIOS     = copiar('municipalities_legal_amazon.shp')
AMAZONIA_LEGAL = copiar('states_legal_amazon.shp')

# (arquivo, camada) — None = camada única
RECENTE = [
    (copiar('yearly_deforestation_biome_amazonia_v20261016.gpkg'), None),
    (copiar('cerrado_2008_2025.gpkg'), 'cerrado_2008_2025'),
]
HISTORICO = [
    (copiar('accumulated_deforestation_2007_biome_amazonia_v20261016.gpkg'), None),
    (copiar('cerrado_mascara_2000_recortado_.gpkg'), None),
    (copiar('cerrado_2001_2007_recortado.gpkg'), None),
]
ANOS = list(range(2008, 2026))
CRS_AREA = 'ESRI:102033'  # South America Albers Equal Area
CRS_SAIDA = 'EPSG:4674'   # SIRGAS 2000, para uso no QGIS

## 3. Teste de integridade, camadas e campo de ano

In [ ]:
def campo_ano(f, layer):
    campos = list(pyogrio.read_info(f, layer=layer)['fields'])
    for c in campos:
        if c.lower() in ('year', 'ano'):
            return c
    raise KeyError(f'Campo de ano não encontrado em {os.path.basename(f)}. Campos: {campos}')

erros = []
for f, layer in [(MUNICIPIOS, None), (AMAZONIA_LEGAL, None)] + RECENTE + HISTORICO:
    try:
        camadas = [c[0] for c in pyogrio.list_layers(f)]
        if layer is None and len(camadas) > 1:
            erros.append(f'{f}: várias camadas {camadas}, especifique qual'); st = 'CAMADA?'
        else:
            pyogrio.read_info(f, layer=layer); st = 'OK     '
    except Exception:
        st = 'ERRO   '; erros.append(f)
    print(st, f'{os.path.getsize(f)/1e6:7.1f} MB', os.path.basename(f), layer or '')
if erros:
    raise RuntimeError(erros)
CAMPO_ANO = {f: campo_ano(f, l) for f, l in RECENTE}
print('Campos de ano:', {os.path.basename(k): v for k, v in CAMPO_ANO.items()})

## 4. Municípios e limite da Amazônia Legal

In [ ]:
al = gpd.read_file(AMAZONIA_LEGAL, engine='pyogrio').to_crs(CRS_AREA)
al['geometry'] = shapely.make_valid(al.geometry.to_numpy())
al = al.dissolve()[['geometry']]

mun = gpd.read_file(MUNICIPIOS, engine='pyogrio').to_crs(CRS_AREA)
mun['geometry'] = shapely.make_valid(mun.geometry.to_numpy())
COD, NOME, UF = 'geocodigo', 'nome', 'UF'
mun[UF] = mun[COD].astype(str).str[:2].map({
    '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA',
    '16': 'AP', '17': 'TO', '21': 'MA', '51': 'MT'})
mun = gpd.overlay(mun[[COD, NOME, UF, 'geometry']], al, how='intersection', keep_geom_type=True)
mun = mun.dissolve(by=[COD, NOME, UF]).reset_index()
mun['area_mun_km2'] = mun.area / 1e6
print(len(mun), 'municípios na Amazônia Legal')

mg = mun.geometry.to_numpy()
shapely.prepare(mg)
tree = shapely.STRtree(mg)
N = len(mun)

## 5. Funções de leitura e interseção

In [ ]:
def ler(f, layer, cols=None):
    cols = cols or []
    g = gpd.read_file(f, layer=layer, engine='pyogrio', use_arrow=True, columns=cols).to_crs(CRS_AREA)
    g = g.explode(index_parts=False, ignore_index=True)
    geoms = g.geometry.to_numpy()
    inval = ~shapely.is_valid(geoms)
    if inval.any():
        geoms[inval] = shapely.make_valid(geoms[inval])
    ok = ~shapely.is_empty(geoms) & ~shapely.is_missing(geoms)
    attrs = g.loc[ok, cols].reset_index(drop=True) if cols else None
    return geoms[ok], attrs

def pares(d):
    idx_d, idx_m = tree.query(d, predicate='intersects')
    dentro = shapely.contains(mg[idx_m], d[idx_d])
    a = shapely.area(d[idx_d])
    fora = ~dentro
    if fora.any():
        a[fora] = shapely.area(shapely.intersection(d[idx_d[fora]], mg[idx_m[fora]]))
    return idx_d, idx_m, a / 1e6, fora.sum()

## 6. Desmatamento recente 2008–2025 (série anual)

In [ ]:
serie = np.zeros((N, len(ANOS)))
for f, layer in RECENTE:
    t = time.time()
    col = CAMPO_ANO[f]
    d, attrs = ler(f, layer, [col])
    ano = pd.to_numeric(attrs[col], errors='coerce').to_numpy()
    idx_d, idx_m, a, nf = pares(d)
    ano_p = ano[idx_d]
    ok = (ano_p >= ANOS[0]) & (ano_p <= ANOS[-1])
    if (~ok).any():
        print(f'  aviso: {a[~ok].sum():,.1f} km² com ano fora de 2008–2025 ignorados '
              f'(anos: {sorted(set(np.unique(ano_p[~ok]).tolist()))[:10]})')
    k = idx_m[ok] * len(ANOS) + (ano_p[ok] - ANOS[0]).astype(int)
    serie += np.bincount(k, weights=a[ok], minlength=N * len(ANOS)).reshape(N, len(ANOS))
    print(f'  {os.path.basename(f)}: {len(d):,} polígonos, {nf:,} cruzando divisas, {time.time()-t:.0f}s')
    del d, attrs, idx_d, idx_m, a

## 7. Desmatamento histórico até 2007

In [ ]:
hist = np.zeros(N)
for f, layer in HISTORICO:
    t = time.time()
    d, _ = ler(f, layer)
    idx_d, idx_m, a, nf = pares(d)
    hist += np.bincount(idx_m, weights=a, minlength=N)
    print(f'  {os.path.basename(f)}: {len(d):,} polígonos, {nf:,} cruzando divisas, {time.time()-t:.0f}s')
    del d, idx_d, idx_m, a

## 8. Tabelas por município, estado e Amazônia Legal

In [ ]:
res = mun[[COD, NOME, UF, 'area_mun_km2']].set_index(COD)
res['historico_ate_2007_km2'] = hist
res['recente_2008_2025_km2'] = serie.sum(axis=1)
res['total_km2'] = res['historico_ate_2007_km2'] + res['recente_2008_2025_km2']
anual = pd.DataFrame(serie, index=res.index, columns=[str(a) for a in ANOS])

print(f"Amazônia Legal: {res['total_km2'].sum():,.0f} km² "
      f"(histórico {res['historico_ate_2007_km2'].sum():,.0f}, recente {res['recente_2008_2025_km2'].sum():,.0f})")

cols = ['area_mun_km2', 'historico_ate_2007_km2', 'recente_2008_2025_km2', 'total_km2']
def pct(df):
    df = df.copy()
    for c in cols[1:]:
        df[c.replace('_km2', '_pct')] = 100 * df[c] / df['area_mun_km2']
    return df.round(2)

tab_mun = pct(res).sort_values([UF, NOME])
tab_uf  = pct(res.groupby(UF)[cols].sum())
tab_al  = pct(res[cols].sum().to_frame('Amazônia Legal').T)

anual_mun = res[[NOME, UF]].join(anual.round(3)).sort_values([UF, NOME])
anual_uf  = anual.join(res[UF]).groupby(UF).sum().round(2)
anual_al  = anual.sum().round(2).to_frame('Amazônia Legal').T

## 9. Exportação: planilha e CSV

In [ ]:
REN = {COD: 'Código IBGE', NOME: 'Município', 'area_mun_km2': 'Área total (km²)',
       'historico_ate_2007_km2': 'Desmatado até 2007 (km²)', 'historico_ate_2007_pct': 'Desmatado até 2007 (%)',
       'recente_2008_2025_km2': 'Desmatado 2008–2025 (km²)', 'recente_2008_2025_pct': 'Desmatado 2008–2025 (%)',
       'total_km2': 'Desmatado total (km²)', 'total_pct': 'Desmatado total (%)'}
ORDEM = ['Área total (km²)', 'Desmatado até 2007 (km²)', 'Desmatado até 2007 (%)',
         'Desmatado 2008–2025 (km²)', 'Desmatado 2008–2025 (%)',
         'Desmatado total (km²)', 'Desmatado total (%)']
def fmt(df, idx):
    df = df.reset_index().rename(columns={'index': idx}).rename(columns=REN)
    return df[[c for c in df.columns if c not in ORDEM] + ORDEM]
def fmt_anual(df, idx):
    return df.reset_index().rename(columns={'index': idx}).rename(columns=REN)

with pd.ExcelWriter(OUT + 'desmatamento_areas.xlsx') as w:
    fmt(tab_mun, 'Código IBGE').to_excel(w, sheet_name='Municípios', index=False)
    fmt(tab_uf, 'UF').to_excel(w, sheet_name='Estados (parte na AL)', index=False)
    fmt(tab_al, 'Recorte').to_excel(w, sheet_name='Amazônia Legal', index=False)
    fmt_anual(anual_mun, 'Código IBGE').to_excel(w, sheet_name='Anual - Municípios', index=False)
    fmt_anual(anual_uf, 'UF').to_excel(w, sheet_name='Anual - Estados', index=False)
    fmt_anual(anual_al, 'Recorte').to_excel(w, sheet_name='Anual - Amazônia Legal', index=False)

tab_mun.reset_index().to_csv(OUT + 'municipios.csv', index=False)
tab_uf.reset_index().to_csv(OUT + 'estados.csv', index=False)
(anual.join(res[[NOME, UF]]).reset_index()
      .melt(id_vars=[COD, NOME, UF], var_name='ano', value_name='desmatamento_km2')
      .assign(ano=lambda x: x['ano'].astype(int))
      .sort_values([UF, NOME, 'ano'])
      .to_csv(OUT + 'serie_anual_municipios.csv', index=False))

## 10. Exportação: GeoPackage

In [ ]:
gpkg = OUT + 'desmatamento_amazonia_legal.gpkg'
if os.path.exists(gpkg): os.remove(gpkg)
anual_cols = anual.add_prefix('a').round(3)
g_mun = mun[[COD, 'geometry']].set_index(COD).join(tab_mun).join(anual_cols).reset_index()
g_mun = gpd.GeoDataFrame(g_mun, geometry='geometry', crs=CRS_AREA).to_crs(CRS_SAIDA)
g_mun.to_file(gpkg, layer='municipios', driver='GPKG', engine='pyogrio')

g_uf = mun[[UF, 'geometry']].dissolve(by=UF).join(tab_uf).join(anual_cols.join(res[UF]).groupby(UF).sum().round(2))
g_uf = g_uf.reset_index().to_crs(CRS_SAIDA)
g_uf.to_file(gpkg, layer='estados', driver='GPKG', engine='pyogrio')

display(fmt(tab_al, 'Recorte'), fmt_anual(anual_al, 'Recorte'))

## 11. Compactação, cópia para o Drive e download

In [ ]:
zipf = '/content/resultados_desmatamento.zip'
with zipfile.ZipFile(zipf, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        z.write(OUT + f, f)
shutil.copy(zipf, DRIVE)
for f in os.listdir(OUT):
    print(f'{os.path.getsize(OUT + f)/1e6:6.1f} MB  {f}')
files.download(zipf)